In [ ]:
from pathlib import Path

dataset = Path(r'D:\Project\dataset')

In [ ]:
IfilenameNum = dataset/Path(r'pdf\numble-from1to100.pdf')
IfilenameDirection = dataset/Path(r'pdf\marke-in-angel.pdf')
Ofilename = dataset/Path(r'pdf\out-numble-from1to100.pdf')

In [ ]:
from pypdf import PdfReader, PdfWriter

reader = PdfReader(IfilenameNum)
writer = PdfWriter()
meta = reader.metadata

print(len(reader.pages))

In [ ]:
#rotate
for page in reader.pages:
    writer.add_page(page.rotate(90))
with open(Ofilename, 'wb') as fp:
    writer.write(fp)

In [ ]:
#merge
writer.add_page(reader.pages[0])
writer.pages[0].merge_page(reader.pages[1])

with open(Ofilename, 'wb') as fp:
    writer.write(fp)

In [ ]:
#scale_by

page = reader.pages[0]
print('before: ',page.artbox)
page.scale_by(2)
print('after: ', page.artbox)
writer.add_page(page)

with open(Ofilename, 'wb') as fp:
    writer.write(fp)

In [ ]:
#translate
from pypdf import Transformation

pageZero = reader.pages[0]
pageOne = reader.pages[1]

transformation = Transformation().translate(tx=-200)
pageZero.add_transformation(transformation)
pageZero.merge_page(pageOne)
writer.add_page(pageZero)

with open(Ofilename, 'wb') as fp:
    writer.write(fp)

In [ ]:
#PaperSize
from pypdf import PaperSize
papersize = PaperSize
print('papersize')
print('A3:', papersize.A3)
print('A4:', papersize.A4)

In [ ]:
#rotate and translate
from pypdf import PaperSize, PdfWriter

writer = PdfWriter(clone_from=dataset/Path(r'pdf/black.pdf'))
writer.add_blank_page(PaperSize.A0.width, PaperSize.A0.height)

transformation = Transformation().rotate(45).translate(tx=595+595,ty=421)
writer.pages[1].merge_page(writer.pages[0])
writer.pages[1].merge_transformed_page(writer.pages[0], transformation)

writer.write(Ofilename)

In [ ]:
#rotate and translate
from pypdf import PaperSize, PdfWriter

writer = PdfWriter(clone_from=dataset/Path(r'pdf/arrow.pdf'))
writer.add_blank_page(PaperSize.A3.width, PaperSize.A3.height)

transformation = Transformation().rotate(90).translate(tx=842,ty=0)
writer.pages[1].merge_transformed_page(writer.pages[0], transformation)

transformation = Transformation().rotate(90).translate(tx=842,ty=595)
writer.pages[1].merge_transformed_page(writer.pages[0], transformation)
writer.write(Ofilename)

In [141]:
page = writer.pages[0]
def show_boxes(page):
    print('mediabox: ',page.mediabox)
    print('cropbox: ', page.cropbox)
    print('trimbox: ', page.trimbox)
    print('bleedbox: ', page.bleedbox)
    print('artbox: ', page.artbox)
    print('---+---+---+---+')
page.scale_by(2)
print('after scale_by(2) ')
show_boxes(page)

page.scale(sx=0.5,sy=2)
print('after scale(sx=0.5,sy=2): ')
show_boxes(page)
print('writer.pages[0].artbox.top:', writer.pages[0].artbox.top)


after scale_by(2) 
mediabox:  RectangleObject([0.0, 0.0, 1190.552, 6735.12])
cropbox:  RectangleObject([0.0, 0.0, 1190.552, 6735.12])
trimbox:  RectangleObject([0.0, 0.0, 1190.552, 6735.12])
bleedbox:  RectangleObject([0.0, 0.0, 1190.552, 6735.12])
artbox:  RectangleObject([0.0, 0.0, 1190.552, 6735.12])
---+---+---+---+
after scale(sx=0.5,sy=2): 
mediabox:  RectangleObject([0.0, 0.0, 595.276, 13470.24])
cropbox:  RectangleObject([0.0, 0.0, 595.276, 13470.24])
trimbox:  RectangleObject([0.0, 0.0, 595.276, 13470.24])
bleedbox:  RectangleObject([0.0, 0.0, 595.276, 13470.24])
artbox:  RectangleObject([0.0, 0.0, 595.276, 13470.24])
---+---+---+---+
writer.pages[0].artbox.top: 13470.24


In [138]:
from pypdf import PaperSize, PdfWriter

writer = PdfWriter(clone_from=dataset/Path(r'pdf/arrow.pdf'))
writer.add_blank_page(PaperSize.A3.height, PaperSize.A3.width)

transformation = Transformation()
writer.pages[1].merge_transformed_page(writer.pages[0], transformation)

transformation = Transformation().translate(tx=595,ty=0)
writer.pages[1].merge_transformed_page(writer.pages[0], transformation)
writer.write(Ofilename)

(True, <_io.FileIO [closed]>)

In [ ]:
#booklet
Ifilename = dataset/Path(r'pdf/2.pdf')
reader = PdfReader(Ifilename)
writer = PdfWriter()

transformation = Transformation().translate(tx=595,ty=0)


for page in reader.pages:
    page.scale(sx=(PaperSize.A4.width/page.artbox.right), sy=(PaperSize.A4.height/page.artbox.top))
    

pages_len = len(reader.pages)
if pages_len % 2 == 0:
    is_pages_odd = 1
else:
    is_pages_odd = 0

for _ in range(int( (pages_len+1) / 2 )):
    writer.add_blank_page(PaperSize.A3.height, PaperSize.A3.width)

if is_pages_odd:
    for n in range(int(pages_len/2)):
        if n % 2 == 0:
            writer.pages[n].merge_transformed_page(reader.pages[n], transformation)
            writer.pages[n].merge_page(reader.pages[pages_len-1-n])
        else:
            writer.pages[n].merge_page(reader.pages[n])
            writer.pages[n].merge_transformed_page(reader.pages[pages_len-1-n], transformation)
else:
    writer.pages[0].merge_transformed_page(reader.pages[0], transformation)
    for n in range(int((pages_len-1)/2)):
        n+=1
        if n % 2 == 0:
            writer.pages[n].merge_transformed_page(reader.pages[n], transformation)
            writer.pages[n].merge_page(reader.pages[pages_len-1-n])
        else:
            writer.pages[n].merge_page(reader.pages[n])
            writer.pages[n].merge_transformed_page(reader.pages[pages_len-1-n], transformation)
writer.write(Ofilename)


(True, <_io.FileIO [closed]>)

In [ ]:
Ifilename = dataset/Path(r'pdf/2.pdf')
reader = PdfReader(Ifilename)
writer = PdfWriter()

transformation = Transformation().translate(tx=595,ty=0)


for page in reader.pages:
    page.scale(sx=(PaperSize.A4.width/page.artbox.right), sy=(PaperSize.A4.height/page.artbox.top))
    

pages_len = len(reader.pages)
if pages_len % 2 == 0:
    is_pages_odd = 1
else:
    is_pages_odd = 0

for _ in range(int( (pages_len+1) / 2 )):
    writer.add_blank_page(PaperSize.A3.height, PaperSize.A3.width)

if is_pages_odd:
    for n in range(int(pages_len/2)):
        if n % 2 == 0:
            writer.pages[n].merge_transformed_page(reader.pages[n], transformation)
            writer.pages[n].merge_page(reader.pages[pages_len-1-n])
        else:
            writer.pages[n].merge_page(reader.pages[n])
            writer.pages[n].merge_transformed_page(reader.pages[pages_len-1-n], transformation)
else:
    writer.pages[0].merge_transformed_page(reader.pages[0], transformation)
    for n in range(int((pages_len-1)/2)):
        n+=1
        if n % 2 == 0:
            writer.pages[n].merge_transformed_page(reader.pages[n], transformation)
            writer.pages[n].merge_page(reader.pages[pages_len-1-n])
        else:
            writer.pages[n].merge_page(reader.pages[n])
            writer.pages[n].merge_transformed_page(reader.pages[pages_len-1-n], transformation)
writer.write(Ofilename)
